# ECHO — the signal that was never sent
**Entangled Minds: The Quantathon · Qubits Qrew · NED University**
Case file QX-01 · Modules 01–06 (all solved: 3 mandatory + 3 stretch)

**How to read this notebook:** every module follows the same rhythm — **Concept → Prediction → Run → What it proves**. Each result is also logged to an *evidence board* (FACT / CLAIM / INFERENCE / OPEN QUESTION) that is printed at the end.

> **Source of the circuits.** The seed circuit (Module 02) and the teleportation circuit (Module 06) are the organisers' recovered circuits, run unchanged first; my own extended analysis follows.

> 🧠 **AI-assistance note.** This notebook was drafted with AI assistance (allowed by the Code of Conduct §2). Before the live round, be able to explain every cell in your own words — the judges will change a circuit and ask you to predict the result. A judge-prep cheat sheet is at the end.

In [ ]:
# 0 · Setup — safe to re-run
!pip install qiskit qiskit-aer pylatexenc matplotlib numpy --quiet

import math
import numpy as np
import matplotlib.pyplot as plt
from statistics import NormalDist

import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace
from qiskit.visualization import plot_histogram, plot_bloch_multivector

sim = AerSimulator()
print("Qiskit", qiskit.__version__)

def run(circuit, shots=1024, seed=None):
    # Run a circuit `shots` times; return the counts dict. (Optional seed -> reproducible.)
    kw = {"seed_simulator": seed} if seed is not None else {}
    return sim.run(transpile(circuit, sim), shots=shots, **kw).result().get_counts()

# ---- Evidence board: every module logs here, printed at the end -------------
board = []
def log(kind, module, text):
    assert kind in ("FACT", "CLAIM", "INFERENCE", "OPEN QUESTION")
    board.append((kind, module, text))

# ---- Small stats helpers (stdlib only) --------------------------------------
def p_hat_ci(k, n, conf=0.95):
    # Wilson score interval for a probability estimated as k successes in n trials.
    z = NormalDist().inv_cdf(0.5 + conf/2); p = k/n
    centre = (p + z*z/(2*n)) / (1 + z*z/n)
    half = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return p, centre-half, centre+half

def chi2_fair_pvalue(n0, n1):
    # Chi-square test (1 dof) of 'this bit is 50/50'. Returns (chi2, p-value).
    n = n0 + n1; e = n/2
    chi2 = (n0-e)**2/e + (n1-e)**2/e
    return chi2, math.erfc(math.sqrt(chi2/2))

print("Ready.")

---
# MODULE 01 · The Gatekeeper  *(mandatory)*
*Gates available: H · X · measure. One and two qubits.*

**Concept.** A qubit starts in |0⟩. **X** flips |0⟩↔|1⟩ (a NOT). **H** (Hadamard) turns |0⟩ into (|0⟩+|1⟩)/√2, an equal superposition: measuring gives 0 or 1 with probability ½ each. Measurement samples a classical bit; we repeat many *shots* to see probabilities.

### 🥉 Bronze — "always 1" and "perfectly uncertain"
**Prediction:** `X` → always 1. `H` → ≈50/50 (the wobble around 500/1000 is ~±16, i.e. √(n·¼)).

In [ ]:
# Bronze: bit that is ALWAYS 1
q_always1 = QuantumCircuit(1, 1)
q_always1.x(0)
q_always1.measure(0, 0)
c_always1 = run(q_always1, 1000)

# Bronze: bit that is PERFECTLY UNCERTAIN
q_coin = QuantumCircuit(1, 1)
q_coin.h(0)
q_coin.measure(0, 0)
c_coin = run(q_coin, 1000)

print("X  ->", c_always1)
print("H  ->", c_coin)
display(q_always1.draw("mpl")); display(q_coin.draw("mpl"))
plot_histogram([c_always1, c_coin], legend=["X (always 1)", "H (50/50)"]);

log("FACT", "M01", f"X then measure gave {c_always1} over 1000 shots (deterministic).")
log("FACT", "M01", f"H then measure gave {c_coin} over 1000 shots (≈50/50).")

**What it proves:** a gate can make a measurement outcome certain (X) or maximally uncertain (H). Without gates the qubit would just be a boring classical 0.

### 🥈 Silver — "looks uncertain but is secretly certain"
**Sequence:** `H` then `H`.  **Prediction:** always 0.

**Why (amplitudes, not probabilities):** H|0⟩ = (|0⟩+|1⟩)/√2. Apply H again:
(H|0⟩ + H|1⟩)/√2 = ½(|0⟩+|1⟩) + ½(|0⟩−|1⟩) = |0⟩. The two paths to |1⟩ have opposite signs (+½ and −½) and **cancel (destructive interference)**; the paths to |0⟩ add (constructive). H is its own inverse. If you had used *probabilities* (½ then ½ again) you'd predict 50/50 — amplitudes carry sign (phase); probabilities don't. That is the whole point.

In [ ]:
q_hh = QuantumCircuit(1, 1)
q_hh.h(0); q_hh.h(0)
q_hh.measure(0, 0)
c_hh = run(q_hh, 1000)
print("H,H ->", c_hh)

# Show the amplitudes at each step (before measuring)
s0 = Statevector.from_label("0")
s1 = s0.evolve(QuantumCircuit(1)); qa = QuantumCircuit(1); qa.h(0)
s_after_h = s0.evolve(qa)
qb = QuantumCircuit(1); qb.h(0); qb.h(0)
s_after_hh = s0.evolve(qb)
print("after one H :", np.round(s_after_h.data, 4))
print("after two H :", np.round(s_after_hh.data, 4))

log("FACT", "M01", f"H,H then measure gave {c_hh}: looks random after one H but is deterministic after two.")
log("INFERENCE", "M01", "Two H gates cancel because amplitudes interfere (+½ and −½ paths to |1⟩ cancel): quantum states carry phase, not just probabilities.")

### 🥇 Gold — the vault code `101`, built from `000`
Use X on qubits 0 and 2 (leave qubit 1 alone). Qiskit prints bit-strings with the *highest-index bit on the left*, but `101` is a palindrome so the order cannot be mistaken — still, we build it deliberately.

In [ ]:
vault = QuantumCircuit(3, 3)      # starts as |000>
vault.x(2)                        # leftmost printed bit  = qubit 2 -> 1
vault.x(0)                        # rightmost printed bit = qubit 0 -> 1
vault.measure([0, 1, 2], [0, 1, 2])
c_vault = run(vault, 1000)
print("Vault ->", c_vault)
display(vault.draw("mpl"))
assert list(c_vault.keys()) == ["101"], "vault code wrong!"
log("FACT", "M01", f"Vault pattern: X on qubits 0 and 2, measured -> {c_vault} (101 in every shot).")

**One-line proof (M01):** Gates set the probability amplitudes — X gives a certain 1, H a 50/50 bit, H·H cancels back to a certain 0 by interference, and X on qubits 0 and 2 builds the vault code 101.

---
# MODULE 02 · The Honest Coin  *(stretch — solved)*
**Question:** ECHO was triggered by a "random" quantum seed. Should Zara have trusted it?

### 🥉 Bronze — a genuine quantum coin, shown fair
A single H + measure. "Fair" is never *proven* by a finite sample — we can only say the data are **consistent with** 50/50, and bound how far off it could be (confidence interval) and test the hypothesis p = ½ (chi-square).

In [ ]:
def analyse_bits(counts, label="circuit", conf=0.95):
    # For every classical bit position: P(1), a confidence interval, and a fairness test.
    counts = {k.replace(" ", ""): v for k, v in counts.items()}   # drop register separators
    n = sum(counts.values())
    width = len(next(iter(counts)))
    out = []
    print(f"=== {label}: {n} shots, counts = {dict(sorted(counts.items()))}")
    for pos in range(width):                       # pos 0 = rightmost = classical bit 0
        n1 = sum(v for k, v in counts.items() if k[::-1][pos] == "1")
        n0 = n - n1
        p, lo, hi = p_hat_ci(n1, n, conf)
        chi2, pv = chi2_fair_pvalue(n0, n1) if n0 and n1 else (float("inf"), 0.0)
        print(f"  bit {pos}: P(1) = {p:.4f}   {int(conf*100)}% CI = [{lo:.4f}, {hi:.4f}]   "
              f"chi2 = {chi2:.2f}   p-value = {pv:.4g}")
        out.append((pos, p, lo, hi, pv))
    return out

def quantum_coin(shots=1000):
    qc = QuantumCircuit(1, 1); qc.h(0); qc.measure(0, 0)
    return run(qc, shots)

coin_counts = quantum_coin(1000)
res_coin = analyse_bits(coin_counts, "quantum coin (H)")
fair = res_coin[0][4] > 0.05
print("Consistent with a fair coin at the 5% level?", fair)

# How does the estimate tighten with more shots?
for n in (100, 1000, 10000, 100000):
    c = quantum_coin(n); p, lo, hi = p_hat_ci(c.get("1", 0), n)
    print(f"{n:>6} shots: P(1) = {p:.4f}, 95% CI half-width = {(hi-lo)/2:.4f}")

log("FACT", "M02", f"Quantum coin (H) over 1000 shots: {coin_counts}; P(1)={res_coin[0][1]:.3f}, 95% CI [{res_coin[0][2]:.3f},{res_coin[0][3]:.3f}], chi-square p={res_coin[0][4]:.3f}.")
log("INFERENCE", "M02", "Data are consistent with a fair coin; we cannot prove exact fairness, only bound the bias (CI half-width ≈ 1/√n).")

### 🥈 Silver — run the organisers' **seed circuit** and tell the truth
> **Seed circuit (organisers' original, `ry(1.159)`).** Run unchanged, as instructed.

Two independent ways to find the truth:
1. **Exact:** compute the true output probabilities from the statevector (no sampling error).
2. **Empirical:** sample many shots and test with the CI/chi-square above.

In [ ]:
# --- 1) The organisers' recovered seed circuit, run exactly as given ---
seed_original = QuantumCircuit(1, 1)
seed_original.ry(1.159, 0)
seed_original.measure(0, 0)
print("Original seed circuit, 2000 shots:", run(seed_original, 2000))

# --- 2) Same circuit without the measurement, for exact analysis ---
def seed_circuit():
    qc = QuantumCircuit(1)
    qc.ry(1.159, 0)
    return qc

seed = seed_circuit()
display(seed.draw("mpl"))

# 1) exact distribution
exact = Statevector(seed).probabilities_dict()
print("Exact probabilities:", {k: round(v, 6) for k, v in exact.items()})

# 2) empirical
seed_m = seed.copy()
seed_m.measure_all()
seed_counts = run(seed_m, 20000)
res_seed = analyse_bits(seed_counts, "SEED CIRCUIT (ry 1.159)")
plot_histogram(seed_counts, title="Seed circuit — measured");

In [ ]:
# Verdict, written by the data rather than by us
n = sum(seed_counts.values())
if len(seed_counts) == 1:
    verdict_seed = "DETERMINISTIC: the 'random' seed always returns the same value — it is not random at all."
else:
    worst = min(res_seed, key=lambda r: r[4])       # most suspicious bit
    pos, p, lo, hi, pv = worst
    if pv < 0.01:
        verdict_seed = (f"BIASED: bit {pos} has P(1)={p:.3f} (95% CI [{lo:.3f},{hi:.3f}], p={pv:.2g}). "
                        f"It *looks* random but is detectably not 50/50 — a 'quantum' seed is not automatically a fair one.")
    else:
        verdict_seed = (f"CONSISTENT WITH FAIR at this sample size (p={pv:.2f}); any remaining bias is bounded by the CI "
                        f"[{lo:.3f},{hi:.3f}].")
print("VERDICT:", verdict_seed)

log("FACT", "M02", "Seed circuit result: " + verdict_seed)
log("FACT", "M02", f"Exact seed probability P(1) = sin²(1.159/2) = {exact.get('1', 0):.4f} (P(0) = {exact.get('0', 0):.4f}).")
log("INFERENCE", "M02", f"The angle 1.159 equals 2·arcsin(√0.30) = {2*math.asin(math.sqrt(0.30)):.4f} rad: the seed is tuned to a round 30/70 split. An accidental miscalibration would be unlikely to land on a round probability, so the bias looks DESIGNED (an inference, not proof of who or why).")
log("INFERENCE", "M02", "Random-LOOKING ≠ random. 'Quantum' only guarantees randomness if the state really is a balanced superposition; the seed's distribution must be measured, not assumed. Also, whoever chooses the circuit chooses the seed's distribution — a human-trust issue physics cannot fix.")

### 🥇 Gold — put a number on any bias, and how many shots to be *sure*
You can never be 100% *sure*; you pick a false-alarm rate α and a detection power 1−β. For a coin with true P(1) = ½+δ, testing "is it fair?" at significance α (two-sided) with power 1−β needs

$$ n \;\approx\; \left(\frac{z_{1-\alpha/2}\,\sqrt{p_0 q_0} \;+\; z_{1-\beta}\,\sqrt{p_1 q_1}}{\delta}\right)^2,\qquad p_0=\tfrac12,\; p_1=\tfrac12+\delta. $$

Rule of thumb: the standard error is 0.5/√n, so **to resolve a bias of δ you need about n ≈ 1/δ² shots** (95% confidence). An 8-point bias (δ=0.08) needs ~150 shots; a 1-point bias needs ~10,000; 0.1 point needs ~1,000,000.

We also check the "unlucky day" probability directly: how often does a *genuinely fair* coin fail the test?

In [ ]:
def shots_needed(delta, alpha=0.05, power=0.95):
    z = NormalDist().inv_cdf
    p0, p1 = 0.5, 0.5 + delta
    num = z(1-alpha/2)*math.sqrt(p0*(1-p0)) + z(power)*math.sqrt(p1*(1-p1))
    return math.ceil((num/delta)**2)

print(f"{'bias δ':>8} | {'shots (95% power)':>18} | {'rule of thumb 1/δ²':>20}")
for d in (0.10, 0.05, 0.02, 0.01, 0.005, 0.001):
    print(f"{d:>8} | {shots_needed(d):>18,} | {round(1/d**2):>20,}")

# Measured bias of the seed circuit and the shots needed to detect *it*
p1_seed = exact.get("1", 0.0)
delta_seed = abs(p1_seed - 0.5)
if delta_seed > 0:
    print(f"\nSeed circuit: true P(1) = {p1_seed:.4f} -> bias δ = {delta_seed:.4f} -> "
          f"≈ {shots_needed(delta_seed):,} shots to detect it with 95% power.")
else:
    print("\nSeed circuit has no first-order bias on bit 0 (δ=0).")

# 'Unlucky day' check: a FAIR coin, tested at alpha=5% — how often is it wrongly flagged?
rng = np.random.default_rng(7)
for n in (100, 1000, 10000):
    k = rng.binomial(n, 0.5, size=20000)
    chi2 = 2*((k-n/2)**2/(n/2)); pv = np.array([math.erfc(math.sqrt(c/2)) for c in chi2])
    print(f"n={n:>6}: fair coin wrongly flagged in {np.mean(pv<0.05)*100:.1f}% of 20,000 trials (expected ≈ 5%)")

log("INFERENCE", "M02", "Detecting bias δ needs ≈1/δ² shots; the false-alarm ('fair coin, unlucky day') rate equals the α we choose (5% here, shown by simulation). Certainty is impossible — only confidence.")
log("OPEN QUESTION", "M02", "Even a perfectly fair seed does not show ECHO fired 'by chance': who chose the circuit/threshold that turned the seed into a trigger?")

**One-line proof (M02):** The recovered seed is not a fair coin: its exact P(1) is 0.30, so ECHO's "random" trigger was biased — chance alone did not decide when it fired.

---
# MODULE 03 · The Two Observers  *(mandatory · the heart of it)*
**The claim:** twin detectors always agreed ⇒ "proof they were linked across space."
**The crux of the case:** *agreement alone does not prove a quantum link.*

### 🥉 Bronze — the pair that only ever agrees (Bell state)
`H` on qubit 0 then `CNOT(0→1)` gives (|00⟩+|11⟩)/√2. **Prediction:** only `00` and `11`, ≈50/50 between them, never `01`/`10`.

In [ ]:
bell = QuantumCircuit(2, 2)
bell.h(0)
bell.cx(0, 1)
bell.measure([0, 1], [0, 1])
c_bell = run(bell, 4000)
print("Bell pair (Z,Z):", c_bell)
display(bell.draw("mpl")); display(plot_histogram(c_bell))
assert "01" not in c_bell and "10" not in c_bell

agree_bell = (c_bell.get("00", 0) + c_bell.get("11", 0)) / 4000
log("FACT", "M03", f"Bell pair measured in Z,Z: {c_bell}; agreement = {agree_bell:.3f} (no 01/10).")

### 🥈 Silver — a *pre-agreed secret* also always agrees
Two people share a secret coin flip **beforehand** (they flip once, copy the result, walk apart — *no communication afterwards*). Each reads out the secret. They agree 100% of the time, exactly like the Bell pair. Statistics: identical.

⇒ **"Always agreeing" does NOT prove a quantum link.** It is equally explained by a *local hidden variable* ("shared secret"). No message, no spooky action needed.

In [ ]:
# Classical shared-secret model: flip ONE coin before they separate, each simply reads it out.
rng = np.random.default_rng(1)
secret = rng.integers(0, 2, size=4000)
classical_counts = {}
for s in secret:
    key = f"{s}{s}"                         # Alice reads s, Bob reads s
    classical_counts[key] = classical_counts.get(key, 0) + 1
print("Classical shared secret:", classical_counts)
print("Quantum Bell pair (Z,Z):", c_bell)
plot_histogram([c_bell, classical_counts], legend=["Quantum Bell pair", "Classical shared secret"]);

agree_cl = (classical_counts.get("00", 0) + classical_counts.get("11", 0)) / 4000
log("FACT", "M03", f"A classical pre-agreed secret also gives agreement = {agree_cl:.3f} and the same 00/11 histogram.")
log("INFERENCE", "M03", "In the Z basis the quantum pair and the pre-agreed secret are statistically indistinguishable ⇒ 'the detectors always agreed' is NOT evidence of a quantum link, still less of a message.")

### 🥇 Gold — an experiment that tells them apart
The trick: **don't measure the same way every time.** Two levels of test, from simple to airtight.

**Level A — change the basis.** To measure in the X-basis, apply `H` before measuring. For the Bell state |Φ⁺⟩:
- Z⊗Z: agree always (shown). X⊗X: **also agree always.** Mixed Z⊗X: **no correlation (50/50).**

A *single* shared coin flip cannot do this: it gives Z-agreement, but then has nothing to say in the X basis (its X outcomes would be independent → only 50% agreement). Level A therefore kills the *simplest* secret. But a cleverer secret could pre-store answers for **both** bases, so we need:

**Level B — CHSH (Bell test).** Alice picks one of two measurement angles (a₀=0, a₁=π/2), Bob one of two (b₀=π/4, b₁=−π/4) — *independently, no communication*. Compute
S = E(a₀,b₀) + E(a₀,b₁) + E(a₁,b₀) − E(a₁,b₁), with E = P(agree) − P(disagree).
- **Any** pre-agreed secret / local hidden variable obeys **|S| ≤ 2** (proof below: S = a₀(b₀+b₁) + a₁(b₀−b₁), and one bracket is 0, the other ±2).
- The quantum Bell pair gives **S = 2√2 ≈ 2.83**.

(Measuring along an angle θ in the X–Z plane = apply `ry(−θ)` then measure Z. This uses one rotation beyond the module's H/X/CNOT gate set — fine for a stretch-style proof; it is exactly the point of the experiment.)

In [ ]:
# ---------- Level A: change the basis ----------
def bell_pair_basis(basis_a, basis_b, shots=4000):
    qc = QuantumCircuit(2, 2)
    qc.h(0); qc.cx(0, 1)
    if basis_a == "X": qc.h(0)       # measure qubit 0 in X basis
    if basis_b == "X": qc.h(1)       # measure qubit 1 in X basis
    qc.measure([0, 1], [0, 1])
    return run(qc, shots)

def agree(counts):
    n = sum(counts.values()); return (counts.get("00", 0) + counts.get("11", 0)) / n

levelA = {}
for ba, bb in [("Z","Z"), ("X","X"), ("Z","X")]:
    c = bell_pair_basis(ba, bb); levelA[(ba, bb)] = agree(c)
    print(f"Bell pair, Alice {ba} / Bob {bb}: agreement = {agree(c):.3f}   {c}")

# One shared coin: Z outcomes equal the secret, X outcomes are independent coin flips
rng = np.random.default_rng(3)
def secret_pair(ba, bb, shots=4000):
    s = rng.integers(0, 2, shots)
    a = np.where(np.full(shots, ba == "Z"), s, rng.integers(0, 2, shots))
    b = np.where(np.full(shots, bb == "Z"), s, rng.integers(0, 2, shots))
    return float(np.mean(a == b))
for ba, bb in [("Z","Z"), ("X","X"), ("Z","X")]:
    print(f"1-coin secret, Alice {ba} / Bob {bb}: agreement = {secret_pair(ba, bb):.3f}")

In [ ]:
# ---------- Level B: CHSH ----------
def E_chsh(theta_a, theta_b, shots=8000):
    qc = QuantumCircuit(2, 2)
    qc.h(0); qc.cx(0, 1)             # |Phi+>
    qc.ry(-theta_a, 0)               # Alice measures along angle theta_a
    qc.ry(-theta_b, 1)               # Bob   measures along angle theta_b
    qc.measure([0, 1], [0, 1])
    c = run(qc, shots)
    same = c.get("00", 0) + c.get("11", 0); diff = c.get("01", 0) + c.get("10", 0)
    return (same - diff) / shots

a0, a1 = 0.0, math.pi/2
b0, b1 = math.pi/4, -math.pi/4
E00, E01, E10, E11 = E_chsh(a0, b0), E_chsh(a0, b1), E_chsh(a1, b0), E_chsh(a1, b1)
S_quantum = E00 + E01 + E10 - E11
print(f"E(a0,b0)={E00:+.3f}  E(a0,b1)={E01:+.3f}  E(a1,b0)={E10:+.3f}  E(a1,b1)={E11:+.3f}")
print(f"S (quantum Bell pair) = {S_quantum:.3f}    [theory 2√2 = {2*math.sqrt(2):.3f}]")

# Best any local hidden variable / pre-agreed secret can do: enumerate all 16 deterministic strategies
best = max(A0*B0 + A0*B1 + A1*B0 - A1*B1
           for A0 in (-1, 1) for A1 in (-1, 1) for B0 in (-1, 1) for B1 in (-1, 1))
print(f"Max S over ALL deterministic pre-agreed strategies = {best}  (mixing strategies cannot exceed this)")

# Plot
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(["Classical limit\n(any secret)", "Quantum Bell pair\n(measured)", "Quantum max\n2√2"], [2, S_quantum, 2*math.sqrt(2)],
       color=["#999", "#c8903a", "#7a6"])
ax.axhline(2, ls="--", c="k"); ax.set_ylabel("CHSH S"); ax.set_title("Bell test: the pre-agreed secret cannot exceed 2")
plt.tight_layout(); plt.show()

# Statistical significance (conservative): each E has SE <= 1/sqrt(n); four independent terms -> SE(S) <= 2/sqrt(n)
n = 8000; se_S = 2*math.sqrt(1/n)                # conservative bound
print(f"conservative standard error of S ≤ {se_S:.3f}  ->  violation is {(S_quantum-2)/se_S:.0f}σ above the classical bound")

log("FACT", "M03", f"Level A (Bell pair): ZZ agreement {levelA[('Z','Z')]:.2f}, XX agreement {levelA[('X','X')]:.2f}, mixed ZX agreement {levelA[('Z','X')]:.2f}.")
log("FACT", "M03", f"CHSH on the Bell pair: S = {S_quantum:.3f} (classical limit 2, quantum max 2.828).")
log("INFERENCE", "M03", "S>2 is impossible for any pre-agreed secret ⇒ the pair really is entangled: the correlations are stronger than any classical secret can explain. This is the first real crack in ECHO: ECHO's data would show agreement, but agreement alone could never have established the link.")
log("OPEN QUESTION", "M03", "Zara's logs only show Z-basis agreement (blue/blue, red/red). Did she ever run a Bell test? If not, her 'proof of linkage' was never tested against the pre-agreed-secret explanation.")

### Bonus (the crux): entanglement ≠ a communication channel — the **no-signalling test**
Even though the pair is genuinely entangled, **Bob's own results look 50/50 no matter what Alice does.** Whatever Alice does to her half — nothing, flip it, measure it in Z, measure in X — Bob's *marginal* statistics don't move. The correlations only show up when you **compare both lists**, and comparing needs an ordinary classical channel. That is why entanglement can't carry a message faster than light.

In [ ]:
def bob_marginal(alice_action, shots=6000):
    qc = QuantumCircuit(2, 2)
    qc.h(0); qc.cx(0, 1)
    if alice_action == "X on her qubit":      qc.x(0)
    if alice_action == "H on her qubit":      qc.h(0)
    if alice_action == "measure in Z":        qc.measure(0, 0)
    if alice_action == "measure in X":        qc.h(0); qc.measure(0, 0)
    qc.measure(1, 1)
    c = run(qc, shots)
    n1 = sum(v for k, v in c.items() if k[0] == "1")       # leftmost char = classical bit 1 = Bob
    return n1 / shots

actions = ["do nothing", "X on her qubit", "H on her qubit", "measure in Z", "measure in X"]
print("Alice's action          ->  P(Bob reads 1)")
bob_ps = {}
for a in actions:
    bob_ps[a] = bob_marginal(a); print(f"  {a:<22} ->  {bob_ps[a]:.3f}")
print("\nBob's statistics never change: Alice cannot encode a message in what Bob sees.")
log("FACT", "M03", "Bob's marginal P(1) ≈ 0.50 for every action Alice takes: " + ", ".join(f"{a}: {p:.2f}" for a, p in bob_ps.items()))
log("INFERENCE", "M03", "No-signalling: entanglement gives correlation, not a communication channel. A message needs a classical channel to compare results.")

**One-line proof (M03):** Perfect agreement is reproduced by a pre-agreed secret, so it proves nothing alone; only the CHSH value S ≈ 2.83 > 2 shows true entanglement, and Bob's data never depends on Alice's actions, so no message can be sent.

---
# MODULE 04 · The Erased Measurement  *(stretch — solved)*
*Gates: H · X · measure. Single qubit.*
**Question:** a log entry "changed when it was observed." ARIA blames the investigators for corrupting it. Physics — or a cover story?

**Concept.** Gates are **unitary = reversible** (you can always undo them by running the inverse). **Measurement is not** reversible: it collapses the state and the phase information is lost for good. "Observing changes the result" is real physics — but only if the observation happens *in the middle*, and it changes the result in a *predictable* way.

### 🥉 Bronze — recoverable vs destroyed
- **Recoverable:** `H` then `H` (the inverse of H is H) — the original |0⟩ comes back with certainty.
- **Destroyed:** `H`, **measure in the middle**, then `H` — the intermediate measurement collapsed the state, and the second H can no longer undo it: result is 50/50.

**The difference:** *without* a mid-way measurement amplitudes interfere and cancel (information preserved, reversible); *with* it, the state collapses to |0⟩ or |1⟩ (a classical bit) and the interference that restored |0⟩ is gone.

In [ ]:
# Recoverable: H, H
q_rec = QuantumCircuit(1, 1)
q_rec.h(0); q_rec.h(0); q_rec.measure(0, 0)
c_rec = run(q_rec, 2000)

# Destroyed: H, MEASURE (mid-circuit), H, measure
q_des = QuantumCircuit(1, 2)
q_des.h(0)
q_des.measure(0, 0)          # the "observer" looks mid-way
q_des.h(0)
q_des.measure(0, 1)
c_des_full = run(q_des, 2000)
# final-bit marginal (classical bit 1 is the left char)
c_des = {"0": 0, "1": 0}
for k, v in c_des_full.items(): c_des[k[0]] += v

print("H,H            ->", c_rec, "   (recoverable: always 0)")
print("H,[look],H     ->", c_des, "  (destroyed: 50/50)")
display(q_rec.draw("mpl")); display(q_des.draw("mpl"))
plot_histogram([c_rec, c_des], legend=["H,H (unobserved)", "H,[measure],H (observed)"]);

log("FACT", f"M04", f"H,H -> {c_rec}; H,measure,H -> {c_des}. Observing mid-way destroyed the deterministic outcome.")

### 🥈 Silver — predict first, then run
| Sequence | Prediction (reasoning) |
|---|---|
| **X, H, H** | X: |0⟩→|1⟩. H·H = identity ⇒ stays |1⟩ ⇒ **always 1** |
| **H, X, H** | H X H = **Z** (Z gate). Z|0⟩ = |0⟩ ⇒ **always 0** |

*What's really doing the "erasing":* it is **not** the gates "deleting data" — it is **interference**. In H,X,H the first H creates a superposition, X swaps the two amplitudes (which, for |+⟩ = (|0⟩+|1⟩)/√2, changes nothing: |+⟩ is an X-eigenstate), and the second H interferes them back into |0⟩. The "information" was never lost — it was hiding in the *relative phase*, which a measurement *in the middle* would destroy.

I'll use a tiny exact simulator (2×2 matrices) as my "predict first" tool, then confirm with Qiskit.

In [ ]:
# --- Exact predictor: multiply 2x2 matrices. This is the 'predict BEFORE you run it' tool. ---
GATE = {
    "x": np.array([[0, 1], [1, 0]], dtype=complex),
    "h": np.array([[1, 1], [1, -1]], dtype=complex) / math.sqrt(2),
    "z": np.array([[1, 0], [0, -1]], dtype=complex),
    "i": np.eye(2, dtype=complex),
}

def predict(seq):
    # seq like 'xhh' or ['x','h','m','h']; 'm' = mid-circuit measurement (collapse).
    # Returns (final probabilities [P0, P1], note). Uses the density matrix so 'm' = dephasing.
    rho = np.array([[1, 0], [0, 0]], dtype=complex)
    for g in seq:
        if g == "m":
            rho = np.diag(np.diag(rho))              # measurement: kill the off-diagonals (coherence)
        else:
            U = GATE[g]; rho = U @ rho @ U.conj().T
    return np.real(np.diag(rho))

def run_seq(seq, shots=4000):
    n_mid = seq.count("m")
    qc = QuantumCircuit(1, n_mid + 1)
    k = 0
    for g in seq:
        if g == "m": qc.measure(0, k); k += 1
        else: getattr(qc, g if g != "i" else "id")(0)
    qc.measure(0, n_mid)
    c = run(qc, shots)
    n1 = sum(v for key, v in c.items() if key[0] == "1")
    return np.array([1 - n1/shots, n1/shots])

for seq in ["xhh", "hxh", "hh", "hmh", "xhx", "hzh"]:
    p = predict(seq); r = run_seq(seq)
    print(f"{seq:>5}: predicted P(0),P(1) = {np.round(p, 3)}   measured = {np.round(r, 3)}   "
          f"{'✅' if np.allclose(p, r, atol=0.05) else '❌'}")

In [ ]:
log("FACT", "M04", "X,H,H -> always 1; H,X,H -> always 0 (both confirmed by Qiskit and by hand: HXH = Z).")
log("INFERENCE", "M04", "The 'erasing' is interference of amplitudes (relative phase), not data being deleted. Unitary gates are reversible; measurement is not.")

### 🥇 Gold — be ready for the judge's new sequence
Use the **predictor** above plus these rules (memorise the logic, not the table):

1. **Gates are 2×2 matrices; read circuits left→right, multiply right→left.**
2. **XX = HH = ZZ = I.**  **HXH = Z.**  **HZH = X.** (H swaps the Z and X "bases".)
3. Start |0⟩. **Z on |0⟩ does nothing visible**; Z only matters via interference with a later H.
4. **A mid-circuit measurement** replaces the state by a *classical coin* with the same Z-probabilities: it deletes the phase. After it, anything that depended on interference is gone.
5. **Final probabilities** = |amplitude|².

Below: random sequences, prediction vs. result — and what *you* do when the judge hands you one: write it as `predict("...")`, say the answer out loud, then run it.

In [ ]:
import random
random.seed(11)
rows = []
for _ in range(8):
    seq = "".join(random.choice("xhzm") for _ in range(random.randint(3, 6)))
    p = predict(seq); r = run_seq(seq, 3000)
    ok = np.allclose(p, r, atol=0.05)
    rows.append(ok)
    print(f"{seq:>7}: predict {np.round(p, 2)}  run {np.round(r, 2)}  {'✅' if ok else '❌'}")
print(f"\n{sum(rows)}/{len(rows)} predictions matched.")
assert all(rows)
log("FACT", "M04", f"Predictor (hand-built 2×2 matrix model with a measurement/dephasing rule) matched Qiskit on {sum(rows)}/{len(rows)} random sequences.")
log("INFERENCE", "M04", "ARIA's 'it changed when you observed it' is consistent with REAL physics — but only for a mid-way observation, and it explains a 50/50 scramble, not a specific altered value. It cannot excuse a log whose content was *altered in a controlled way*.")
log("OPEN QUESTION", "M04", "Was the log entry scrambled randomly (physics) or rewritten to a chosen value (tampering)? A physics-only 'observer effect' predicts a 50/50 outcome, not a selected one.")

**One-line proof (M04):** A measurement in the middle destroys the phase (H, measure, H gives 50/50) while unobserved gates are reversible (H·H gives 0), so ARIA's "it changed when observed" explains a random scramble, not a chosen edit.

---
# MODULE 05 · The Damaged Message  *(mandatory)*
Each original bit was sent three times; at most one of every three flipped. **Majority vote** (3-bit repetition code).

### 🥉 Bronze — recover the message

In [ ]:
raw = "100011001001001011100110001011000000001100011011000111000000110000001000000111001010101011111111"
stream = [raw[i:i+3] for i in range(0, len(raw), 3)]       # 32 triples
assert len(stream) == 32 and all(len(t) == 3 for t in stream)

def majority(t): return "1" if t.count("1") >= 2 else "0"

decoded_bits = "".join(majority(t) for t in stream)
print("Decoded bits:", decoded_bits, f"({len(decoded_bits)} bits)")
chars = [chr(int(decoded_bits[i:i+8], 2)) for i in range(0, len(decoded_bits), 8)]
message = "".join(chars)
print("Message (ASCII):", message)

# Which triples were damaged, and where?
for i, t in enumerate(stream):
    maj = majority(t)
    bad = [j for j, b in enumerate(t) if b != maj]
    if bad: print(f"  triple {i:>2} = {t}  -> {maj}   flipped position {bad[0]}")
n_err = sum(1 for t in stream if t not in ("000", "111"))
print(f"\n{n_err}/32 triples damaged; {32-n_err} untouched. All have ≤1 flip, so majority vote is guaranteed correct here.")
log("FACT", "M05", f"Majority-vote decoding of the 96-bit stream gives the 32 bits {decoded_bits} = ASCII '{message}'. {n_err} of 32 triples had exactly one flipped bit.")

**Which value do you trust?** The majority of the three. Reason: if at most one bit flips, the two that agree must be right (a flip can't create two matching wrong bits).


### 🥈 Silver — where majority vote **fails**
It fails when **two or more bits** in the same triple flip: e.g. sent `000` (meaning 0) arrives as `011` → majority says **1** — a *confident wrong answer*, and the decoder has no idea. Corruption of 2 in a triple looks identical to a clean `1`.

In [ ]:
def encode(bits):  return ["".join(b*3) for b in bits]
def decode(triples): return "".join(majority(t) for t in triples)

original = "01000101"                    # 'E'
sent = encode(original)

def flip(triple, positions):
    t = list(triple)
    for p in positions: t[p] = "1" if t[p] == "0" else "0"
    return "".join(t)

print("Original bits :", original, "->", chr(int(original, 2)))
for n_flips in (0, 1, 2, 3):
    damaged = sent.copy()
    damaged[1] = flip(damaged[1], list(range(n_flips)))          # corrupt triple #1 (true bit = 1)
    dec = decode(damaged)
    status = "✅ correct" if dec == original else "❌ WRONG (silently!)"
    print(f"{n_flips} flip(s) in triple 1: {sent[1]} -> {damaged[1]}  decoded = {dec}   {status}")

log("FACT", "M05", "Counter-example: a triple 111 corrupted to 001 (2 flips) decodes to 0 instead of 1 — majority vote is confidently wrong at ≥2 flips per triple.")

**Exactly where it breaks:** at the **2nd flip inside one triple**. 0 flips → right; 1 flip → right (corrected); 2 flips → *wrong, undetected*; 3 flips → wrong, undetected (looks like a clean opposite bit).

### 🥇 Gold — correct vs notice, and why you can't "just copy a qubit three times"
**Correct vs notice** (distance d = 3, because 000 and 111 differ in 3 places):
- It can **correct** ⌊(d−1)/2⌋ = **1** flip (per triple).
- It can **detect (notice, without fixing)** d−1 = **2** flips: if the triple isn't 000/111 you know *something* flipped.
- ≥ 3 flips (all three) turns 000 into 111: **undetectable**. And with 2 flips, if you *insist on correcting* you correct to the wrong value.
*(Trade-off: use the code to correct 1 **or** to detect 2 — not both.)*

**Why a qubit can't just be copied three times — the no-cloning theorem.** There is no operation that takes an unknown state |ψ⟩ to |ψ⟩|ψ⟩|ψ⟩. A CNOT "copies" only basis states |0⟩/|1⟩ — on a superposition it creates **entanglement**, not copies. Also, "looking at the three copies to take a vote" would be a measurement that destroys the superposition. The quantum fix **encodes** a|0⟩+b|1⟩ → a|000⟩+b|111⟩ (spread, not copied) and finds errors by **parity checks** that don't reveal a or b.

In [ ]:
# --- No-cloning demo: CNOT 'copying' of |+> ---------------------------------------
qc = QuantumCircuit(3); qc.h(0); qc.cx(0, 1); qc.cx(0, 2)          # attempt to copy |+> onto q1,q2
tried = Statevector(qc)
want = Statevector.from_label("+++")
fid = abs(np.vdot(want.data, tried.data))**2
print(f"Fidelity of CNOT-'copy' of |+> with three true copies |+++> = {fid:.3f}   (1.0 would be a perfect copy)")
print("State actually produced:", np.round(tried.data, 3), " <- GHZ state (entangled), not |+++>")

# --- Quantum 3-qubit bit-flip code: protects against ONE X error ------------------
def prep_psi(qc, q):                      # an arbitrary 'unknown' state
    qc.ry(1.3, q); qc.rz(0.5, q)

def bitflip_trial(error_qubits):
    qc = QuantumCircuit(3, 1)
    prep_psi(qc, 0)
    qc.cx(0, 1); qc.cx(0, 2)                 # ENCODE: a|0>+b|1> -> a|000>+b|111>
    for q in error_qubits: qc.x(q)           # the channel flips these qubits
    qc.cx(0, 1); qc.cx(0, 2); qc.ccx(1, 2, 0)   # DECODE + correct (majority vote, coherently)
    prep_inv = QuantumCircuit(1); prep_psi(prep_inv, 0)
    qc.compose(prep_inv.inverse(), qubits=[0], inplace=True)   # undo the prep: perfect recovery <=> |0>
    qc.measure(0, 0)
    c = run(qc, 2000)
    return c.get("0", 0) / 2000

print("\nQuantum bit-flip code — P(recover the unknown state):")
for errs in [[], [0], [1], [2], [0, 1]]:
    print(f"  X errors on qubits {errs!s:<8}: {bitflip_trial(errs):.3f}")
print("1 flip is always corrected, 2 flips fail — the same d=3 limits as the classical code.")
log("FACT", f"M05", f"CNOT 'copy' of |+> has fidelity {fid:.2f} with |+++> (it makes a GHZ state instead). A 3-qubit bit-flip code recovers an unknown state perfectly from 1 flip but not from 2.")
log("INFERENCE", "M05", "Classical redundancy (copying) is impossible for unknown qubits (no-cloning); quantum error correction spreads information by entanglement and checks parity. This bit-flip code only guards against X errors, not phase (Z) errors — the full Shor code is needed for both.")

**One-line proof (M05):** Majority vote decodes the stream to "ECHO" because each triple has at most one flip, but it silently fails at two flips, and a qubit cannot be protected by copying (no-cloning).

---
# MODULE 06 · The Impossible Claim  *(stretch — solved)*
**Zara's claim:** she *teleported* a message between two stations with no qubit crossing the gap.
**Quantum teleportation protocol** (3 qubits): q0 = the state |ψ⟩ to send (Alice), q1 = Alice's half of a Bell pair, q2 = Bob's half.
1. Make the Bell pair (q1,q2).  2. Alice does CNOT(q0→q1), H(q0).  3. Alice **measures q0, q1** → two ordinary classical bits (m₀, m₁).  4. She **sends the two bits to Bob** over a normal channel.  5. Bob applies **X^m₁** then **Z^m₀** to q2 → q2 = |ψ⟩.

> The organisers' recovered circuit is run **unchanged** first (message state = Ry(1.0)|0⟩). My own generalised version (any state, plus exact density-matrix checks) follows.

In [ ]:
# --- The organisers' recovered teleportation circuit, run unchanged ---
a = 1.0
def teleport(with_correction=True):
    q=QuantumRegister(3); c0=ClassicalRegister(1,'c0'); c1=ClassicalRegister(1,'c1'); co=ClassicalRegister(1,'co')
    qc=QuantumCircuit(q,c0,c1,co)
    qc.ry(a,q[0])
    qc.h(q[1]); qc.cx(q[1],q[2])
    qc.cx(q[0],q[1]); qc.h(q[0])
    qc.measure(q[0],c0); qc.measure(q[1],c1)
    if with_correction:
        with qc.if_test((c1,1)): qc.x(q[2])
        with qc.if_test((c0,1)): qc.z(q[2])
    qc.ry(-a,q[2]); qc.measure(q[2],co)
    return qc

def receiver_accuracy(counts, shots=2000):
    return round(100*sum(v for k,v in counts.items() if k[0]=='0')/shots)

acc_with, acc_without = receiver_accuracy(run(teleport(True))), receiver_accuracy(run(teleport(False)))
print(f"Original circuit — receiver accuracy WITH classical bits: {acc_with}%   WITHOUT: {acc_without}%")
log("FACT", "M06", f"Organisers' circuit: receiver gets the state {acc_with}% of the time with the two classical bits applied, {acc_without}% without (a coin flip).")


def prep(qc, q, theta, phi):                 # the 'message' state |psi> = Rz(phi) Ry(theta) |0>
    qc.ry(theta, q); qc.rz(phi, q)

def teleport_circuit(theta, phi, correction=True, verify=True):
    # Build the teleportation circuit with REAL classical bits and classically-controlled corrections.
    # correction=False models 'the two classical bits never arrive'.
    qr_n = 3
    c_m0 = ClassicalRegister(1, "m0")      # result of measuring q0  -> controls Z
    c_m1 = ClassicalRegister(1, "m1")      # result of measuring q1  -> controls X
    c_out = ClassicalRegister(1, "out")    # final check on Bob's qubit
    qc = QuantumCircuit(qr_n); qc.add_register(c_m0, c_m1, c_out)
    prep(qc, 0, theta, phi)                # q0 = |psi>
    qc.h(1); qc.cx(1, 2)                   # Bell pair shared by Alice (q1) and Bob (q2)
    qc.barrier()
    qc.cx(0, 1); qc.h(0)                   # Alice's Bell-basis measurement
    qc.measure(0, c_m0[0]); qc.measure(1, c_m1[0])
    qc.barrier()
    if correction:                         # the two classical bits travel to Bob
        with qc.if_test((c_m1, 1)): qc.x(2)
        with qc.if_test((c_m0, 1)): qc.z(2)
    if verify:                             # undo the prep on Bob's qubit: perfect arrival <=> |0>
        undo = QuantumCircuit(1); prep(undo, 0, theta, phi)
        qc.compose(undo.inverse(), qubits=[2], inplace=True)
        qc.measure(2, c_out[0])
    return qc

theta, phi = 1.0, 0.0                      # same message state as the organisers' circuit; try others!
tele = teleport_circuit(theta, phi, correction=True)
display(tele.draw("mpl"))

### 🥉 Bronze — does the state actually arrive?
**Test:** after Bob's corrections, apply the *inverse* of the preparation to his qubit and measure. If |ψ⟩ really arrived, he gets |0⟩ **every time**. (We also do a full-state check with density matrices below.)

In [ ]:
shots = 4000
c_with = run(tele, shots)
p_arrive = sum(v for k, v in c_with.items() if k[0] == "0") / shots      # leftmost char = 'out'
print("With corrections, P(Bob's qubit = |psi>) =", round(p_arrive, 4))

# Full-state, exact check (no sampling): deferred-measurement version of the same protocol
def teleport_statevector(theta, phi, correction=True):
    qc = QuantumCircuit(3)
    prep(qc, 0, theta, phi)
    qc.h(1); qc.cx(1, 2); qc.cx(0, 1); qc.h(0)
    if correction:
        qc.cx(1, 2)             # X^m1 on q2   (coherent version of the classically controlled X)
        qc.cz(0, 2)             # Z^m0 on q2
    return qc

def bloch(rho):
    r = rho.data
    return np.array([2*np.real(r[0, 1]), 2*np.imag(r[1, 0]), np.real(r[0, 0] - r[1, 1])])

target = DensityMatrix(Statevector.from_label("0").evolve(
    (lambda q: (prep(q, 0, theta, phi), q)[1])(QuantumCircuit(1))))
bob = partial_trace(DensityMatrix(teleport_statevector(theta, phi, True)), [0, 1])
fidelity = float(np.real(np.trace(target.data @ bob.data)))
print("Bloch vector sent    :", np.round(bloch(target), 4))
print("Bloch vector at Bob  :", np.round(bloch(bob), 4))
print("Fidelity (exact)     :", round(fidelity, 6))

log("FACT", "M06", f"With the two classical bits applied: P(Bob's qubit = |ψ⟩) = {p_arrive:.3f} (sampled), exact fidelity = {fidelity:.4f}. Teleportation works — when the correction is applied.")

### 🥈 Silver — what if the two classical bits never arrive?
Bob has no idea which of four operations (I, X, Z, XZ) to apply. We **remove the classical correction** and look at what he holds.

**Prediction:** Bob's qubit is a *random mixture of the four possibilities with equal probability*. Mixing |ψ⟩, X|ψ⟩, Z|ψ⟩, XZ|ψ⟩ equally gives the **maximally mixed state I/2** — Bloch vector (0,0,0): a coin flip, *identical for every possible ψ*. Bob learns nothing about |ψ⟩.

In [ ]:
tele_nc = teleport_circuit(theta, phi, correction=False)
c_without = run(tele_nc, shots)
p_arrive_nc = sum(v for k, v in c_without.items() if k[0] == "0") / shots
print("WITHOUT classical bits, P(Bob's qubit = |psi>) =", round(p_arrive_nc, 4), "   (≈0.5 = a coin flip)")

bob_nc = partial_trace(DensityMatrix(teleport_statevector(theta, phi, False)), [0, 1])
print("Bob's density matrix (no correction):\n", np.round(bob_nc.data, 4))
print("Bloch vector at Bob :", np.round(bloch(bob_nc), 4), "  <- (0,0,0): maximally mixed, pure noise")
plot_histogram([c_with, c_without], legend=["bits delivered", "bits NEVER arrive"], title="Does |ψ⟩ arrive at Bob?");
log("FACT", "M06", f"Without the classical correction Bob's qubit is maximally mixed (Bloch vector ≈ 0); P(recovering |ψ⟩) = {p_arrive_nc:.3f} ≈ 0.5.")

### 🥇 Gold — prove nothing useful crossed without the classical channel
**Claim (no-signalling):** with the final correction removed, **Bob's reduced state is I/2 for *every* |ψ⟩ whatsoever.** If his state doesn't depend on ψ, then ψ — the message — cannot be read from it, so no information has reached him. We prove it two ways: (1) numerically for many different states, (2) algebraically.

**Algebra.** Before Alice measures, the full state is ½ Σ_{m₀,m₁} |m₀m₁⟩ ⊗ (X^{m₁}Z^{m₀}|ψ⟩) (up to the standard ordering). Bob's unconditional state = average over the four equally likely outcomes = ¼ Σ σ|ψ⟩⟨ψ|σ† over σ ∈ {I,X,Z,XZ} = I/2 (the "Pauli twirl"). No ψ left. ∎

In [ ]:
rng = np.random.default_rng(42)
print("State sent (Bloch vector)         ->  Bob WITHOUT bits (Bloch)     |  Bob WITH bits (Bloch)")
max_dev = 0
for _ in range(8):
    th, ph = rng.uniform(0, math.pi), rng.uniform(0, 2*math.pi)
    tgt = DensityMatrix(Statevector.from_label("0").evolve((lambda q: (prep(q, 0, th, ph), q)[1])(QuantumCircuit(1))))
    b_nc = partial_trace(DensityMatrix(teleport_statevector(th, ph, False)), [0, 1])
    b_c  = partial_trace(DensityMatrix(teleport_statevector(th, ph, True)),  [0, 1])
    max_dev = max(max_dev, np.max(np.abs(b_nc.data - np.eye(2)/2)))
    print(f"{np.round(bloch(tgt), 2)}  ->  {np.round(bloch(b_nc), 3)}   |  {np.round(bloch(b_c), 2)}")
print(f"\nMax |ρ_Bob(no bits) − I/2| over 8 random states = {max_dev:.2e}  -> independent of ψ")

# Conditional on each classical outcome, Bob's state IS informative — but only if he knows (m0,m1):
print("\nConditional on the 4 outcomes (m0,m1), Bob's states are different rotations of ψ (so the bits carry the 'key'):")
for m0 in (0, 1):
    for m1 in (0, 1):
        qc = QuantumCircuit(3); prep(qc, 0, theta, phi); qc.h(1); qc.cx(1, 2); qc.cx(0, 1); qc.h(0)
        sv = Statevector(qc)
        # project q0=m0, q1=m1 (qiskit: little-endian, state index = q2 q1 q0)
        amps = np.array([sv.data[(b2 << 2) | (m1 << 1) | m0] for b2 in (0, 1)])
        amps = amps / np.linalg.norm(amps)
        bl = bloch(DensityMatrix(amps))
        print(f"  (m0,m1)=({m0},{m1}) -> Bob's Bloch vector = {np.round(bl, 2)}")

log("FACT", "M06", f"For 8 random message states, Bob's state without the classical bits equals I/2 to within {max_dev:.1e} — independent of the message.")
log("INFERENCE", "M06", "Teleportation needs 2 ordinary classical bits sent at ≤ light speed. Entanglement alone carries NO usable information, so ECHO was never faster-than-light: it moved a quantum STATE, but the INFORMATION about how to read it travelled classically.")
log("OPEN QUESTION", "M06", "Did Zara's log include the two classical bits? If she recorded 'success' without them, either the measurement record was hidden/removed (Sana's deleted log?) or she mistook correlation for a received message.")

**One-line proof (M06):** With the two classical bits the receiver gets the state ≈100% of the time, without them ≈50% and Bob's qubit is maximally mixed — nothing useful crossed faster than light.

---
# 🧾 The Evidence Board
Everything we directly measured above is a **FACT**; interpretation is labelled as **INFERENCE**. Claims by suspects are listed separately (we can't test them in a notebook).

In [ ]:
# Suspect claims from the briefing (CLAIM = stated, not verified)
log("CLAIM", "Briefing", "Vance: 'ECHO was never real.'  (publicly called it a fraud for years)")
log("CLAIM", "Briefing", "Sana: 'I only deleted a corrupted log.'  (only person with full records access)")
log("CLAIM", "Briefing", "Bilal: 'I warned her.'  (his report, filed 2 days before, is now missing)")
log("CLAIM", "Briefing", "ARIA: 'No unauthorised command was issued.'  (it ran the schedule and stored logs)")
log("CLAIM", "Briefing", "Zara: 'I proved entanglement can carry a message.'")
log("OPEN QUESTION", "Humans", "Which of the four suspects (if any) altered a log, a seed, or the schedule? Nothing in Modules 01–06 can identify a person — only what the *physics* can and cannot support.")

import textwrap
order = ["CLAIM", "FACT", "INFERENCE", "OPEN QUESTION"]
for kind in ("FACT", "CLAIM", "INFERENCE", "OPEN QUESTION"):
    print("\n" + "="*100 + f"\n{kind}\n" + "="*100)
    for k, m, t in board:
        if k == kind:
            print(textwrap.fill(f"[{m}] {t}", 100, subsequent_indent="      "))

---
# Conclusion
ECHO moved **correlation, not a message**: agreement is reproducible by a shared secret (M03), entanglement gives no channel (M03 bonus), and teleportation needs two classical bits (M06). The "random" seed was tuned to a 30/70 split (M02), so the trigger was configured, not chance. Full reasoning, evidence board, competing theory and limitations are in **CASE_REPORT.md**.